In [ ]:
# IBL dataxtraction

# my_beh_export.py
from __future__ import annotations
import os
from pathlib import Path
import numpy as np
import pandas as pd

from one.api import ONE

# ---------- CONFIG ----------
# Use your local cache only
one = ONE(base_url='https://openalyx.internationalbrainlab.org',cache_dir='//qnap-al001/Data/ibl_brain_wide_map')
SUBFOLDER_NAME = "my_beh"     # subfolder created inside each session
SAVE_FORMATS = ("parquet", "csv")  # choose any of ("parquet","csv")

# ---------- HELPERS ----------
def _cat(series, mapping, categories_order):
    """Map numeric codes to categories safely."""
    s = pd.Series(series, dtype="float64")  # allow NaNs
    mapped = s.map(mapping).astype("category")
    # ensure consistent category order
    mapped = mapped.cat.set_categories(categories_order)
    return mapped

def _get(trials: dict, key: str, fallback=None):
    """Safe get from trials dict, returning fallback (or NaNs) with correct length."""
    if key in trials and trials[key] is not None:
        return trials[key]
    if fallback is not None:
        return fallback
    # try to infer length from any present trials field
    n = len(next(iter(trials.values()))) if trials else 0
    return np.full(n, np.nan)

def build_beh_dataframe(trials: dict) -> pd.DataFrame:
    """
    Convert IBL trials object to a tidy DataFrame with friendly columns.
    Works whether or not Timeline-derived fields are present.
    """
    # Required base fields
    n = len(_get(trials, "intervals"))
    trial_idx = np.arange(1, n + 1)

    # Extract commonly available fields (exist in IBL public releases)
    intervals = _get(trials, "intervals")          # shape (n,2): [start, end]
    go_cue = _get(trials, "goCue_times")
    stim_on = _get(trials, "stimOn_times")
    stim_off = _get(trials, "stimOff_times")
    fb_times = _get(trials, "feedback_times")
    fb_type = _get(trials, "feedbackType")         # usually -1 (unrewarded), +1 (rewarded)
    choice = _get(trials, "choice")                # -1 left, +1 right, 0 no-go
    resp_time = _get(trials, "response_times")
    # contrasts and reward
    cL = _get(trials, "contrastLeft")
    cR = _get(trials, "contrastRight")
    rVol = _get(trials, "rewardVolume")

    # Friendly categorical columns
    choice_cat = _cat(choice, {-1: "Left", 0: "NoGo", 1: "Right"},
                      ["Left", "Right", "NoGo"])
    feedback_cat = _cat(fb_type, {-1: "Unrewarded", 1: "Rewarded"},
                        ["Unrewarded", "Rewarded"])

    df = pd.DataFrame({
        "trialNumber": trial_idx,
        "trialStartTime": intervals[:, 0] if intervals is not None and len(intervals) else np.nan,
        "endTrialTime":   intervals[:, 1] if intervals is not None and len(intervals) else np.nan,
        "stimulusOnsetTime": stim_on,
        "stimulusOffsetTime": stim_off,
        "goCueTime": go_cue,
        "choice": choice_cat,
        "choiceCompleteTime": resp_time,
        "feedback": feedback_cat,
        "feedbackTime": fb_times,
        "rewardVolume": rVol,
        "contrastLeft": cL,
        "contrastRight": cR,
    })

    # Optional “NoTimeline” parity: if stim_on/goCue aren’t present, they’ll be NaN—fine.
    # You can add more columns here if your tasks include them (e.g., block, certainRewardSize, etc.)
    return df

def save_beh_dataframe(df: pd.DataFrame, out_dir: Path, base_name: str = "beh") -> dict[str, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    if "parquet" in SAVE_FORMATS:
        p = out_dir / f"{base_name}.parquet"
        df.to_parquet(p, index=False)
        paths["parquet"] = p
    if "csv" in SAVE_FORMATS:
        p = out_dir / f"{base_name}.csv"
        df.to_csv(p, index=False)
        paths["csv"] = p
    return paths

def session_out_dir(eid: str, inside_session: bool = True, root_out: Path | None = None) -> Path:
    """
    Decide where to create the per-recording folder.
    - inside_session=True -> <sessionPath>/<SUBFOLDER_NAME>
    - otherwise -> <root_out>/<subject>/<YYYY-MM-DD>/<number>/<SUBFOLDER_NAME>
    """
    ses_path = Path(one.eid2path(eid))
    if inside_session:
        return ses_path / SUBFOLDER_NAME
    # Build mirrored structure elsewhere
    subject = ses_path.parts[-4]
    date = ses_path.parts[-3]
    number = ses_path.parts[-2]
    if root_out is None:
        raise ValueError("Provide root_out if inside_session=False.")
    return Path(root_out) / subject / date / number / SUBFOLDER_NAME

def export_one_session(eid: str, inside_session: bool = True, root_out: Path | None = None) -> dict[str, Path]:
    """Load trials for a session, build DataFrame, and save to your folder."""
    # Load all 'trials' arrays in one go (no download in local mode if already present)
    trials = one.load_object(eid, "trials", collection="alf", download_only=False)
    if not trials:
        raise FileNotFoundError(f"No 'trials' object found for {eid}. "
                                "Ensure the session has been extracted locally.")
    df = build_beh_dataframe(trials)
    out_dir = session_out_dir(eid, inside_session=inside_session, root_out=root_out)
    saved = save_beh_dataframe(df, out_dir)
    # Also drop a tiny metadata JSON for traceability
    meta = {
        "eid": eid,
        "session_path": str(one.eid2path(eid)),
        "n_trials": int(len(df)),
        "source": "IBL ALF trials",
        "created_by": "my_beh_export.py",
    }
    (out_dir / "beh_meta.json").write_text(pd.Series(meta).to_json(indent=2))
    return saved



In [2]:
# merge_to_multipage_tiff.py
# Usage: python merge_to_multipage_tiff.py /path/to/folder /path/to/output.tiff
import sys
from pathlib import Path
from PIL import Image


in_dir = r'Z:\HAT009\2025-09-07\TwoP\2025-09-07_t-001'
out_path = r'Z:\HAT009\2025-09-07\TwoP\2025-09-07_t-001\merged.tif'

in_dir = Path(in_dir)
out_path = Path(out_path)

# Collect images (adjust extensions as needed)
exts = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}
files = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in exts])
print(f"Found {len(files)} image files.")

if not files:
    raise SystemExit(f"No images found in: {in_dir}")

pages = []
for fp in files:
    img = Image.open(fp)
    # Ensure a consistent mode for multi-page TIFF (RGB is usually safe)
    if img.mode not in ("L", "RGB", "RGBA"):
        img = img.convert("RGB")
    elif img.mode == "RGBA":
        # Drop alpha to avoid issues with some TIFF viewers
        img = img.convert("RGB")
    pages.append(img)

first, rest = pages[0], pages[1:]

# Save as multi-page TIFF
first.save(
    out_path,
    save_all=True,
    append_images=rest,
    compression="tiff_lzw",   # lossless; try "tiff_deflate" if you prefer
)
print(f"Created multi-page TIFF: {out_path} ({len(pages)} pages)")



Found 14137 image files.


error: argument out of range

In [3]:
len(pages)

14137

In [4]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd

from one.api import ONE
from brainbox.io.one import SpikeSortingLoader
from iblatlas.atlas import Allen

# ---------- CONFIG ----------
ROOT_OUT = Path(r"D:\ibl")         # base output folder
REGION_MAPPING = "Beryl"           # 'Allen' | 'Beryl' | 'Cosmos'
SAVE_BEH_PARQUET_TOO = False       # set True if you also want trials.parquet

# Reuse your authenticated ONE, or uncomment to create here:
one = ONE(base_url='https://openalyx.internationalbrainlab.org',
          cache_dir=str(ROOT_OUT), username='intbrainlab',
          password='international', silent=True)

ba = AllenAtlas()


ImportError: cannot import name '_center' from 'numpy._core.umath' (c:\Users\Huriye\anaconda3\envs\decMaking310\lib\site-packages\numpy\_core\umath.py)

In [ ]:
eids = one.search(
    query_type='remote',
    project='brainwide',
    tag='Brainwidemap',                                     # optional but recommended
    datasets=['_ibl_trials.table.pqt', 'spikes.times.npy'],
    dataset_qc_lte='WARNING'
)
len(eids)

In [ ]:
# ---------- HELPERS ----------
def session_path_from_eid(eid: str) -> Path:
    """Return cache session path (…/<subject>/<YYYY-MM-DD>/<NNN>)."""
    return Path(one.eid2path(eid))

def session_key(eid: str) -> str:
    """<subject>_<YYYY-MM-DD>_<NNN> derived from cache path parts."""
    p = session_path_from_eid(eid)
    subject, date, number = p.parts[-4], p.parts[-3], p.parts[-2]
    return f"{subject}_{date}_{number}"

def session_outdir(eid: str) -> Path:
    d = ROOT_OUT / session_key(eid)
    d.mkdir(parents=True, exist_ok=True)
    return d

def add_stim_side(df: pd.DataFrame) -> pd.DataFrame:
    cl, cr = df.get("contrastLeft"), df.get("contrastRight")
    side = np.where((cr > 0) & ((cl == 0) | pd.isna(cl)), "R",
            np.where((cl > 0) & ((cr == 0) | pd.isna(cr)), "L", "Other"))
    df = df.copy()
    df["stim_side"] = pd.Categorical(side, categories=["L","R","Other"])
    return df

def export_behaviour_csv(eid: str, outdir: Path) -> Path:
    pqt = one.load_dataset(eid, "_ibl_trials.table.pqt")  # downloads if missing
    print(pqt)
    df = pd.read_parquet(pqt, engine="pyarrow")
    df = add_stim_side(df)
    csv_path = outdir / "trials.csv"
    df.to_csv(csv_path, index=False)
    if SAVE_BEH_PARQUET_TOO:
        df.to_parquet(outdir / "trials.parquet", index=False)
    return csv_path

def hemisphere_from_ml(ml: np.ndarray) -> np.ndarray:
    hemi = np.full_like(ml, "", dtype=object)
    hemi[ml < 0] = "LH"
    hemi[ml >= 0] = "RH"
    hemi[np.isnan(ml)] = ""
    return hemi

def clusters_df(spikes: dict, clusters: dict, channels: dict,
                pid: str, eid: str, probe_name: str) -> pd.DataFrame:
    """Tidy cluster table with region/coords/hemisphere."""
    # Ensure channel-derived fields merged into clusters
    ssl_dummy = None  # just keeping signature parity; already merged by caller
    cl = clusters

    nclu = len(cl.get("channels", []))
    if nclu == 0:
        # Fallback: infer from max cluster id
        nclu = int(np.nanmax(spikes["clusters"])) + 1

    mlapdv = cl.get("mlapdv")
    if mlapdv is not None and len(mlapdv) == nclu:
        ml, ap, dv = mlapdv[:, 0], mlapdv[:, 1], mlapdv[:, 2]
    else:
        ml = np.full(nclu, np.nan); ap = np.full(nclu, np.nan); dv = np.full(nclu, np.nan)

    # Region labels (if acronym missing and coords available)
    acr = cl.get("acronym")
    atlas_id = cl.get("atlas_id")
    if (acr is None or atlas_id is None) and mlapdv is not None:
        labs = ba.get_labels(mlapdv, mapping=REGION_MAPPING)
        acr = np.asarray([r["acronym"] for r in labs])
        atlas_id = np.asarray([r["id"] for r in labs])

    counts = np.bincount(spikes["clusters"], minlength=nclu)

    df = pd.DataFrame({
        "eid": eid,
        "pid": pid,
        "probe": probe_name,
        "cluster_id": np.arange(nclu, dtype=int),
        "region": acr if acr is not None else np.array([""] * nclu),
        "atlas_id": atlas_id if atlas_id is not None else np.full(nclu, np.nan),
        "ml": ml, "ap": ap, "dv": dv,
        "hemisphere": hemisphere_from_ml(ml),
        "depth": cl.get("depths", np.full(nclu, np.nan)),
        "n_spikes": counts
    })
    # Optional quality labels
    if "ks2_label" in cl: df["ks2_label"] = cl["ks2_label"]
    if "label" in cl:     df["label"] = cl["label"]
    return df

def channels_df(channels: dict, pid: str, eid: str, probe_name: str) -> pd.DataFrame:
    n = len(channels.get("mlapdv", [])) or len(channels.get("x", []))
    if n == 0:
        return pd.DataFrame()
    mlapdv = channels.get("mlapdv")
    if mlapdv is not None and len(mlapdv):
        ml, ap, dv = mlapdv[:,0], mlapdv[:,1], mlapdv[:,2]
        labs = ba.get_labels(mlapdv, mapping=REGION_MAPPING)
        acr = np.asarray([r["acronym"] for r in labs])
        atlas_id = np.asarray([r["id"] for r in labs])
    else:
        ml = ap = dv = np.full(n, np.nan)
        acr = np.array([""] * n); atlas_id = np.full(n, np.nan)
    return pd.DataFrame({
        "eid": eid, "pid": pid, "probe": probe_name,
        "channel": np.arange(n, dtype=int),
        "ml": ml, "ap": ap, "dv": dv,
        "hemisphere": hemisphere_from_ml(ml),
        "region": acr, "atlas_id": atlas_id
    })


In [ ]:
export_behaviour_csv(eid, outdir)

In [ ]:

eidsAll = eids

eidsSelected = eidsAll[:2]
len(eidsAll), len(eidsSelected)

In [ ]:
eids = eidsSelected
print(f"Exporting {len(eids)} sessions to {ROOT_OUT} …")
for i, eid in enumerate(eids, 1):
    outdir = session_outdir(eid)
    written: list[Path] = []

    # Behaviour
    written.append(export_behaviour_csv(eid, outdir))
    print('CSV created')

    # Probe insertions (spikes + metadata)
    insertions = one.alyx.rest("insertions", "list", session=eid)
    for ins in insertions:
        pid = ins["id"]
        probe_name = ins.get("name") or ins.get("probe_name") or f"probe{pid[:4]}"

        ssl = SpikeSortingLoader(pid=pid, one=one, atlas=ba)
        spikes, clusters, channels = ssl.load_spike_sorting()
        clusters = ssl.merge_clusters(spikes, clusters, channels)

        # 1) spikes
        sp_path = outdir / f"{probe_name}_spikes.npz"
        np.savez_compressed(sp_path, times=spikes["times"], clusters=spikes["clusters"])
        written.append(sp_path)

        # 2) clusters
        dfc = clusters_df(spikes, clusters, channels, pid, eid, probe_name)
        cl_path = outdir / f"{probe_name}_clusters.csv"
        dfc.to_csv(cl_path, index=False)
        written.append(cl_path)

        # 3) channels (optional)
        dfh = channels_df(channels, pid, eid, probe_name)
        if len(dfh):
            ch_path = outdir / f"{probe_name}_channels.csv"
            dfh.to_csv(ch_path, index=False)
            written.append(ch_path)

        # 4) meta
        meta_path = outdir / f"{probe_name}_meta.json"
        meta = {
            "eid": eid, "pid": pid, "probe": probe_name,
            "n_spikes": int(len(spikes["times"])),
            "n_clusters": int(len(dfc)),
            "region_mapping": REGION_MAPPING
        }
        meta_path.write_text(json.dumps(meta, indent=2))
        written.append(meta_path)
        ddfsrfgreg

In [ ]:
df = pd.read_parquet(pqt)

In [ ]:
from brainbox.io.one import SpikeSortingLoader
from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()  # for regions / coordinates

for eid in eids[:5]:  # try a few first
    # --- behaviour (one file)
    trials = one.load_object(eid, 'trials')  # loads trials.table.pqt

    # --- ephys per probe insertion
    # get probe insertion IDs for this session
    pids = [d['id'] for d in one.alyx.rest('insertions','list', session=eid)]
    for pid in pids:
        ssl = SpikeSortingLoader(pid=pid, one=one, atlas=ba)
        spikes, clusters, channels = ssl.load_spike_sorting()  # pulls spikes.* + clusters/channels.* only
        # ...do your analysis here...


In [ ]:
export_one_session(eid, inside_session=False, root_out=Path(r"Z:\my_beh_export"))

# ---------- BATCH EXAMPLES ----------
if __name__ == "__main__":
    # Option A: export a single known session by EID
    # eid = "da8dfec1-d265-44e8-84ce-6ae9c109b8bd"
    # export_one_session(eid)

    # Option B: find local sessions that already have trials (offline friendly)
    # This searches the local cache table; you can narrow by subject/date if you like.
    eids = one.search(dataset="trials")  # returns EIDs that have 'trials' locally
    print(f"Found {len(eids)} sessions with trials.")
    for eid in eids:
        try:
            saved = export_one_session(eid, inside_session=False, root_out=Path(r"Z:\my_beh_export"))
            print(f"✓ {eid} -> {list(saved.values())}")
        except Exception as e:
            print(f"✗ {eid}: {e}")


In [ ]:
# ---- CONFIG ----
one = ONE(base_url='https://openalyx.internationalbrainlab.org',cache_dir='//qnap-al001/Data/ibl_brain_wide_map')
PERF_BINS = [0, 50, 70, 85, 100]   # percent
PERF_LABELS = ["<50", "50–70", "70–85", "85–100"]

# ---- SEARCH: sessions with ephys (+ trials) ----
# 'spikes' => ephys processed present; including 'trials' ensures behaviour exists too
eids = one.search(datasets=['spikes.times.npy', 'trials.table.pqt'], query_type='remote')
print(f"Found {len(eids)} sessions with ephys + trials in local cache")

In [ ]:
one.search_terms()

In [ ]:
# explore_ephys_performance.py


def session_meta_from_path(p: Path) -> dict:
    # session path is .../<subject>/<YYYY-MM-DD>/<001>
    return {
        "subject": p.parts[-4],
        "date": p.parts[-3],
        "number": p.parts[-2],
        "session_path": str(p),
    }

def performance_from_trials(tr: dict) -> tuple[float, float, int]:
    """
    Returns (overall_perf_pct, choice_only_perf_pct, n_trials).
    overall: % trials rewarded (feedbackType==1).
    choice-only: % rewarded among trials with a left/right choice (choice != 0).
    """
    ft = tr.get("feedbackType")
    if ft is None or len(ft) == 0:
        return (np.nan, np.nan, 0)
    ft = np.asarray(ft, dtype=float)
    n_trials = len(ft)
    valid = ~np.isnan(ft)
    overall = (ft[valid] == 1).mean() * 100 if valid.any() else np.nan

    ch = tr.get("choice")
    if ch is not None and len(ch) == n_trials:
        ch = np.asarray(ch, dtype=float)
        mask = valid & (ch != 0)  # exclude NoGo
    else:
        mask = valid
    choice_only = (ft[mask] == 1).mean() * 100 if mask.any() else np.nan
    return overall, choice_only, n_trials

rows = []
for eid in eids:
    try:
        tr = one.load_object(eid, "trials", collection="alf", download_only=False)
        if not tr:
            continue
        overall, choice_only, n_trials = performance_from_trials(tr)
        sp = Path(one.eid2path(eid))
        row = {
            "eid": eid,
            **session_meta_from_path(sp),
            "n_trials": n_trials,
            "perf_overall_pct": overall,
            "perf_choice_only_pct": choice_only,
        }
        rows.append(row)
    except Exception as e:
        print(f"Skipping {eid}: {e}")

df = pd.DataFrame(rows).sort_values(["subject", "date", "number"])
print(f"Computed performance for {len(df)} sessions.")

# ---- Count sessions by performance bands ----
df["perf_bin"] = pd.cut(df["perf_overall_pct"], PERF_BINS, labels=PERF_LABELS, include_lowest=True)
counts = df["perf_bin"].value_counts().sort_index()
print("\nSession counts by overall performance band:")
print(counts)

# Optional: also bin the choice-only metric
df["perf_choice_bin"] = pd.cut(df["perf_choice_only_pct"], PERF_BINS, labels=PERF_LABELS, include_lowest=True)
choice_counts = df["perf_choice_bin"].value_counts().sort_index()
print("\nSession counts by choice-only performance band:")
print(choice_counts)

# Save results
out = Path(r"Z:\ibl_brain_wide_map\_summaries")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "ephys_performance_sessions.csv", index=False)
